# Function 1 - Week 1: planning two or three coverage points

Suppose the next **N evaluations will be used solely to improve coverage**.
The ten supplied locations stay fixed. Where should the N new points go?

The objective is unchanged from the one-point analysis: minimise the average
distance from anywhere in the unit square to the nearest sample, weighting the
whole area equally. The unknown outputs are not used to choose these locations.

In [ ]:
from pathlib import Path
import sys

stage = next(p / "stage-2-bbo" for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "stage-2-bbo/src").is_dir())
sys.path.insert(0, str(stage / "src"))
from initial_exploration import load_initial_data
from batch_coverage_analysis import analyse_coverage_batches, batch_coverage_summary, batch_coordinate_table, batch_strategy_comparison, batch_computation_table, plot_batch_coverage

data = load_initial_data(stage, function_id=1)
output = stage.parent / ".runtime/function-1-batch-coverage"


## What changes when we choose more than one point?

For N = 2, we choose four coordinates together. For N = 3, we choose six.
Each new point competes with the original samples **and the other new points**
to be the nearest sample. Overlapping benefits count only once.

Joint planning lets every proposed point move. Sequential selection instead
chooses the best next point, freezes it, then chooses another. A jointly chosen
pair need not contain the best one-point choice.

The spatial domain remains two-dimensional. We integrate distance over its
nearest-sample regions and search the allowed coordinates from 0 to 0.999999.
These are local calculations, without calls to the course's unknown function.

In [ ]:
study = analyse_coverage_batches(data, output=output)

## How much coverage do we gain?

Every percentage below is measured against the same original ten locations.
The last column shows the extra improvement from allowing one more new point,
in percentage points.

In [ ]:
batch_coverage_summary(study)

Two jointly planned points reduce mean distance by about **14.26%**, compared
with **7.54%** for one point. Three points reach **18.81%**. The additional
improvement gets smaller: roughly 6.73 percentage points for the second point
and 4.55 for the third.

## Where the points go

All panels use the same colour scale. Brighter regions remain farther from a
sample after the proposed points are added. White circles are the fixed original
locations; their numbers retain the output ranks from the first analysis.

Stars mark proposed points. A, B and C identify members of each set, **not a
submission order**. The N = 1, 2 and 3 panels are alternative plans.

In [ ]:
plot_batch_coverage(data, study, save_to=output / "planned-coverage.png")

In [ ]:
batch_coordinate_table(study)

For N = 2, the best numerical plan keeps the one-point choice and adds a point
in the upper-left region. For N = 3, the central point shifts slightly and a
third point enters the lower-right region. The three-point set therefore does
not simply contain the two-point set unchanged.

## Joint planning versus one point at a time

The sequential comparison starts with the same one-point optimum, then
re-optimises each next location while leaving its predecessors fixed. The joint
search can reposition all N new points. Both use the same coverage objective.

In [ ]:
batch_strategy_comparison(study)

For N = 2, the two strategies agree to the displayed precision. For N = 3,
joint planning gains only about **0.006 percentage points** of additional
coverage over sequential selection. The distinction is real, but the practical
gain is small for these particular starting locations.

That agreement for two points is a result of this dataset, not a general rule
that sequential placement is always jointly optimal.

## Is N = 3 too expensive?

**No: all three searches completed on this computer.** The table records the
elapsed time of each search, including its independent starts, local refinement
and checks of neighbouring six-decimal coordinates. Timing varies with machine
load; comparison runs and plotting are separate.

In [ ]:
batch_computation_table(study)

This is not an exhaustive enumeration of every six-decimal coordinate set.
Even restricting candidates to a 41 × 41 grid gives about 1.4 million unordered
pairs and 790 million triples. Instead, two independently seeded
[differential-evolution searches](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.differential_evolution.html)
explore the joint problem, followed by local refinement, including a start from
the sequential plan.

The calculation is still over a two-dimensional square. Six search variables
for three 2D points are different from a function with six input dimensions;
these timings do not establish the cost of coverage search for the later
higher-dimensional functions.

## What the results establish

The independent searches and local refinements agree on these numerical
solutions. The geometric integral is also checked against a separate dense-grid
distance calculation. Point order and duplicate proposals do not change the
score, and overlap between new points is accounted for.

This is strong numerical evidence, not a formal proof of global optimality.
The six-decimal coordinates make the plans reproducible; they do not make the
function's unknown outputs predictable.

The tables are alternative plans for one, two or three of the available queries.
The course still permits one evaluated query per function per week. Final
coverage of a fixed set is independent of submission order, although its
intermediate coverage need not be. Results would affect these plans only if we
choose to use the outputs and change our coverage-only objective.

**No portal queries have been submitted.**